<a href="https://colab.research.google.com/github/CodeWithYokesh/Skin-Cancer/blob/main/ResNet_50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q kaggle
!mkdir -p ~/.kaggle

!mkdir -p ~/.kaggle && echo KGAT_110ef97a4e6ba4f5fc0e5a58561d11c0 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d andrewmvd/isic-2019 -p /content/ --unzip

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/isic-2019
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 9.10G/9.10G [01:42<00:00, 95.4MB/s]



In [3]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# 1. Hardware Device Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {DEVICE}")

# 2. Dataset Paths Configuration
CSV_PATH = 'ISIC_2019_Training_GroundTruth.csv'
IMG_DIR = 'ISIC_2019_Training_Input/ISIC_2019_Training_Input'

# 3. Load Metadata & Filter Target Classes
df = pd.read_csv(CSV_PATH)
def map_label(row):
    if row.get('MEL', 0) == 1: return 0
    elif row.get('BCC', 0) == 1: return 1
    elif row.get('SCC', 0) == 1: return 2
    elif row.get('NV', 0) == 1 or row.get('BKL', 0) == 1 or row.get('DF', 0) == 1 or row.get('VASC', 0) == 1: return 3
    return -1

df['label'] = df.apply(map_label, axis=1)
df = df[df['label'] != -1].reset_index(drop=True)

# 4. Stratified Split (80 / 10 / 10) & Class Weight Calculation
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label'].values
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(DEVICE)

# 5. Dataset & DataLoaders Setup
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx]['image'])
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx]['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(ISICDataset(train_df, IMG_DIR, train_transform), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(ISICDataset(val_df, IMG_DIR, eval_transform), batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(ISICDataset(test_df, IMG_DIR, eval_transform), batch_size=32, shuffle=False, num_workers=0)

# 6. Load Pretrained Wide ResNet-50 Model
print("\nInitializing Wide ResNet-50 Model...")
wrn_model = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT)
in_features = wrn_model.fc.in_features
wrn_model.fc = nn.Linear(in_features, 4)
wrn_model = wrn_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(wrn_model.parameters(), lr=1e-4)

# 7. Training Loop (10 Epochs)
NUM_EPOCHS = 10
best_val_f1 = 0.0

print("\nStarting Wide ResNet-50 Training Loop...")
for epoch in range(NUM_EPOCHS):
    wrn_model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = wrn_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    # Validation Evaluation
    wrn_model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = wrn_model(inputs)
            _, preds = torch.max(outputs, 1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    _, _, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average='weighted')

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Weighted F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(wrn_model.state_dict(), 'wide_resnet50_best.pth')
        print(" -> Saved new best model checkpoint!")

# 8. Test Set Evaluation
print("\n" + "="*45)
print("  EVALUATING WIDE RESNET-50 ON HELD-OUT TEST SET")
print("="*45)

wrn_model.load_state_dict(torch.load('wide_resnet50_best.pth'))
wrn_model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = wrn_model(inputs)
        _, preds = torch.max(outputs, 1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
p, r, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')

print(f"Final Test Accuracy : {test_acc*100:.2f}%")
print(f"Final Test Precision: {p:.4f}")
print(f"Final Test Recall   : {r:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

class_names = ['Melanoma', 'BCC', 'SCC', 'Benign']
print(classification_report(test_labels, test_preds, target_names=class_names, digits=2))

Executing on device: cuda

Initializing Wide ResNet-50 Model...
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-9ba9bcbe.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-9ba9bcbe.pth


100%|██████████| 263M/263M [00:03<00:00, 76.8MB/s]



Starting Wide ResNet-50 Training Loop...
Epoch 1/10 | Train Loss: 0.8100 | Val Weighted F1: 0.7631
 -> Saved new best model checkpoint!
Epoch 2/10 | Train Loss: 0.5408 | Val Weighted F1: 0.7978
 -> Saved new best model checkpoint!
Epoch 3/10 | Train Loss: 0.4263 | Val Weighted F1: 0.7958
Epoch 4/10 | Train Loss: 0.3523 | Val Weighted F1: 0.8106
 -> Saved new best model checkpoint!
Epoch 5/10 | Train Loss: 0.2961 | Val Weighted F1: 0.8064
Epoch 6/10 | Train Loss: 0.2329 | Val Weighted F1: 0.8369
 -> Saved new best model checkpoint!
Epoch 7/10 | Train Loss: 0.2019 | Val Weighted F1: 0.8526
 -> Saved new best model checkpoint!
Epoch 8/10 | Train Loss: 0.1819 | Val Weighted F1: 0.8191
Epoch 9/10 | Train Loss: 0.1714 | Val Weighted F1: 0.8359
Epoch 10/10 | Train Loss: 0.1447 | Val Weighted F1: 0.8457

  EVALUATING WIDE RESNET-50 ON HELD-OUT TEST SET
Final Test Accuracy : 84.10%
Final Test Precision: 0.8528
Final Test Recall   : 0.8410
Final Test F1-Score : 0.8450

              precision  